In [1]:
import os
import urllib.parse
import pandas as pd
from sqlalchemy import create_engine, text

SERVER = r"DANNYPC\SQLEXPRESS"
DATABASE = "MAG7_MarketDB"
DRIVER = "ODBC Driver 17 for SQL Server"
FUNDAMENTALS_DIR = r"C:\Users\Admin\Desktop\Project\data\fundamentals"

params = urllib.parse.quote_plus(
    f"DRIVER={{{DRIVER}}};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    "Trusted_Connection=yes;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    fast_executemany=True
)

print("Connected to:", pd.read_sql("SELECT DB_NAME() AS db_name", engine).iloc[0, 0])

annual_path = os.path.join(FUNDAMENTALS_DIR, "mag7_fundamentals_annual.csv")
quarterly_path = os.path.join(FUNDAMENTALS_DIR, "mag7_fundamentals_quarterly.csv")

annual_df = pd.read_csv(annual_path)
quarterly_df = pd.read_csv(quarterly_path)

df = pd.concat([annual_df, quarterly_df], ignore_index=True)

df["period_end_date"] = pd.to_datetime(df["period_end_date"]).dt.date

out = df[[
    "ticker",
    "frequency",
    "statement",
    "period_end_date",
    "line_item",
    "value"
]].dropna()

with engine.begin() as conn:
    conn.execute(text("DELETE FROM dbo.fundamentals"))

out.to_sql(
    "fundamentals",
    con=engine,
    schema="dbo",
    if_exists="append",
    index=False,
    chunksize=500
)

print(f"Loaded {len(out)} rows into dbo.fundamentals.")
print("Done.")


Connected to: MAG7_MarketDB
Loaded 10344 rows into dbo.fundamentals.
Done.
